# Part 5: incompressible flow through a constriction: implicit, IMEX or multirate?

A pulse of fluid passes a narrow part of a channel and re-accelerates. Near the constriction the mesh is fine and the velocity is five times larger, so an explicit convection step is limited there and nowhere else. This part asks whether that justifies local or multirate time stepping, or whether a fully implicit or IMEX method is the right choice:

1. a small example: the H(div)-conforming HDG discretisation (BDM$_k$ velocity, P$_{k-1}$ pressure, tangential facet unknowns), the pulse at Re 250, ‖div u‖ and the energy balance;
2. the measured comparison on the fine mesh (error at $T$ against wall time, stable steps, cost split, energy), as figures.

The solution is exactly divergence-free, and in 2D such BDM$_k$ fields are the curls of continuous P$_{k+1}$ stream functions. The HDG form is therefore written in $(\psi, \hat u)$: the same discrete velocity, but a symmetric positive definite system without pressure.

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
import numpy as np, time
ngsglobals.msg_level = 0
SetNumThreads(4)

## Geometry

Channel $[0,4]\times[0,1]$, two half-discs of radius 0.4 leave a gap of width 0.2 at $x = 1.5$. Coarse mesh for this page (`maxh` 0.05 for $1 < x < 3$); the measurements below use 0.02 / 0.04 / 0.1.

In [ ]:
L, W, xc, R = 4, 1, 1.5, 0.4
rect = Rectangle(L, W).Face()
rect.edges.Min(X).name, rect.edges.Max(X).name = "inlet", "outlet"
rect.edges.Min(Y).name, rect.edges.Max(Y).name = "wallb", "wallt"
c1, c2 = Circle((xc, 0), R).Face(), Circle((xc, W), R).Face()
c1.edges.name, c2.edges.name = "wallb", "wallt"
dom = rect - c1 - c2
box = MoveTo(xc - 0.5, 0).Rectangle(2, W).Face()
fine = dom * box
fine.faces.maxh = 0.05
mesh = Mesh(OCCGeometry(Glue([dom - box, fine]), dim=2).GenerateMesh(maxh=0.25))
print("elements:", mesh.ne)
Draw(mesh);

## H(div)-HDG in stream-function form

With $u = \operatorname{curl}\psi = (\partial_y\psi, -\partial_x\psi)$, $\nabla u = R\,\nabla^2\psi$ and $|u|^2 = |\nabla\psi|^2$. The viscous form is the HDG form of Lehrenfeld–Schöberl with tangential facet unknowns $\hat u$; the mass is $\int \nabla\psi\cdot\nabla\varphi$. Dirichlet data: $\psi = 0$ on the lower wall, $1$ on the upper wall, $3y^2 - 2y^3$ on the inlet (unit flux, parabolic profile); outlet natural (do-nothing).

In [ ]:
k, nu = 3, 0.004                                  # Re = U_c w_c / nu = 5 * 0.2 / nu = 250
D = "inlet|wallb|wallt"
Hp = H1(mesh, order=k + 1, dirichlet=D)
Vh = TangentialFacetFESpace(mesh, order=k, dirichlet=D)
X = Hp * Vh
(psi, uh), (phi, vh) = X.TnT()
n, h = specialcf.normal(2), specialcf.mesh_size
dS = dx(element_boundary=True)
Rot = CF((0, 1, -1, 0), dims=(2, 2))
tang = lambda w: w - (w * n) * n
u, v = Rot * grad(psi), Rot * grad(phi)
Gu, Gv = Rot * psi.Operator("hesse"), Rot * phi.Operator("hesse")
visc = (InnerProduct(Gu, Gv) * dx + InnerProduct(Gu * n, tang(vh - v)) * dS + InnerProduct(Gv * n, tang(uh - u)) * dS
        + 4 * k * k / h * InnerProduct(tang(vh - v), tang(uh - u)) * dS)
mass = grad(psi) * grad(phi) * dx
M = BilinearForm(mass).Assemble().mat
A = BilinearForm(nu * visc).Assemble().mat

def cond_inverse(form):
    # statically condensed sparse Cholesky, returned as a BaseMatrix on the full space
    bf = BilinearForm(X, condense=True); bf += form; bf.Assemble()
    inv = bf.mat.Inverse(X.FreeDofs(True), "sparsecholesky")
    I = IdentityMatrix(X.ndof)
    return ((I + bf.harmonic_extension) @ inv @ (I + bf.harmonic_extension_trans)) + bf.inner_solve

# steady Stokes flow of the inflow profile: the divergence-free lifting u_S
xS = GridFunction(X)
xS.components[0].Set(mesh.BoundaryCF({"inlet": 3 * y**2 - 2 * y**3, "wallt": 1}, default=0), definedon=mesh.Boundaries(D))
r = xS.vec.CreateVector(); r.data = -BilinearForm(visc).Assemble().mat * xS.vec
xS.vec.data += cond_inverse(visc) * r
print("ndof:", X.ndof)
Draw(Rot * grad(xS.components[0]), mesh, vectors={"grid_size": 40});

The same flow from the mixed form BDM$_k$ $\times$ P$_{k-1}$ (`HDiv`, `L2` with `lowest_order_wb` for condensation, UMFPACK): the velocities agree to rounding.

In [ ]:
V0 = HDiv(mesh, order=k, dirichlet=D)
Q = L2(mesh, order=k - 1, lowest_order_wb=True)
Xm = V0 * Vh * Q
(um, umh, p), (vm, vmh, q) = Xm.TnT()
am = BilinearForm(InnerProduct(Grad(um), Grad(vm)) * dx + InnerProduct(Grad(um) * n, tang(vmh - vm)) * dS
                  + InnerProduct(Grad(vm) * n, tang(umh - um)) * dS + 4 * k * k / h * InnerProduct(tang(vmh - vm), tang(umh - um)) * dS
                  - div(um) * q * dx - div(vm) * p * dx, condense=True).Assemble()
xm = GridFunction(Xm)
xm.components[0].Set(CF((6 * y * (1 - y), 0)), definedon=mesh.Boundaries("inlet"))
r = xm.vec.CreateVector(); am.Apply(xm.vec, r); r.data = -r
r.data += am.harmonic_extension_trans * r
dxm = xm.vec.CreateVector(); dxm.data = am.mat.Inverse(Xm.FreeDofs(True), "umfpack") * r
dxm.data += am.harmonic_extension * dxm; dxm.data += am.inner_solve * r
xm.vec.data += dxm
uS = Rot * grad(xS.components[0])
diff_mixed = sqrt(Integrate((xm.components[0] - uS)**2, mesh)) / sqrt(Integrate(uS**2, mesh))
print(f"|u_mixed - curl psi| / |u| = {diff_mixed:.1e},  ndof mixed {Xm.ndof} vs {X.ndof}")

## The pulse

Inflow $g(t)\,6y(1-y)$ with $g = \sin^2(\pi t)$. The velocity is $u = g(t)\,u_S + \operatorname{curl}\psi$ with $\psi$ vanishing on inlet and walls, so the constraint is homogeneous at every stage and

$$
M\psi' + A(\psi, \hat u) = -g'(t)\, M\psi_S - E^T C(u),
$$

with the upwind DG convection $C$ evaluated on BDM$_k$ (matrix-free, `u.Other()` across facets) and pulled back with the curl embedding $E$ (`ConvertOperator`). **ARS(4,4,3)** (Ascher–Ruuth–Spiteri): Stokes implicit, convection explicit, third order, stiffly accurate, all implicit diagonal entries $\frac12$, so one factorization of $M + \frac{\Delta t}{2}A$ serves every stage. The stages are written with dual vectors,

$$
M Y_s = M\psi^n + \Delta t\sum_{j<s} a_{sj}K_j + \Delta t\sum_{j<s}\hat a_{sj}F_j,\qquad
K_s = \tfrac{2}{\Delta t}\big(MY_s - R_s\big) = -A Y_s - g'(t_s)M\psi_S,\quad F_s = -E^TC(Y_s),
$$

In [ ]:
V = HDiv(mesh, order=k)
E = ConvertOperator(Hp, V, trial_proxy=Hp.TrialFunction(), trial_cf=Rot * grad(Hp.TrialFunction()))
Ex = E @ Embedding(X.ndof, X.Range(0)).T                 # (psi, uhat) -> curl psi in BDM_k
uSV = (Ex * xS.vec).Evaluate()
fS = (M * xS.vec).Evaluate()

w = GridFunction(V)                                      # convecting velocity
gpar = Parameter(0)
ubnd = gpar * IfPos(L / 2 - x, 1, 0) * CF((6 * y * (1 - y), 0))   # inflow data, zero at the outlet
uV, vV = V.TnT()
wn = w * n
conv = BilinearForm(V, nonassemble=True)
conv += -InnerProduct(Grad(vV) * w, uV) * dx
conv += wn * InnerProduct(IfPos(wn, uV, uV.Other(bnd=ubnd)), vV) * dS

g = lambda t: float(np.sin(np.pi * t)**2)
dg = lambda t: float(np.pi * np.sin(2 * np.pi * t))
cV = w.vec.CreateVector()
def C(yv, t, out):                                       # out = E^T C(u), u = g(t) u_S + curl psi
    gpar.Set(g(t)); w.vec.data = g(t) * uSV + Ex * yv
    conv.Apply(w.vec, cV); out.data = Ex.T * cV

# ARS(4,4,3): implicit tableau AI (first column zero), explicit tableau AE, nodes c
AI = [[0, 0, 0, 0, 0], [0, 1/2, 0, 0, 0], [0, 1/6, 1/2, 0, 0], [0, -1/2, 1/2, 1/2, 0], [0, 3/2, -3/2, 1/2, 1/2]]
AE = [[0, 0, 0, 0, 0], [1/2, 0, 0, 0, 0], [11/18, 1/18, 0, 0, 0], [5/6, -5/6, 1/2, 0, 0], [1/4, 7/4, 3/4, -7/4, 0]]
c = [0, 1/2, 2/3, 1/2, 1]

def ars443(dt, T, frames=0):
    S = cond_inverse(mass + dt / 2 * nu * visc)
    yv = xS.vec.CreateVector(); yv[:] = 0
    Y, K, F = ([yv.CreateVector() for _ in range(5)] for _ in range(3))
    R, rhs, yprev = yv.CreateVector(), yv.CreateVector(), yv.CreateVector()
    N = round(T / dt); every = max(1, N // frames) if frames else 0
    hist = []
    for i in range(N):
        t = i * dt
        yprev.data = yv; Y[0].data = yv
        for s in range(5):
            if s > 0:
                R.data = M * yv
                for j in range(s):
                    if AI[s][j]: R.data += dt * AI[s][j] * K[j]
                    if AE[s][j]: R.data += dt * AE[s][j] * F[j]
                rhs.data = R - dt / 2 * dg(t + c[s] * dt) * fS
                Y[s].data = S * rhs
                K[s].data = M * Y[s]; K[s].data -= R; K[s].data *= 2 / dt
            if s < 4:
                C(Y[s], t + c[s] * dt, F[s]); F[s].data *= -1
        yv.data = Y[4]
        if every and (i + 1) % every == 0:
            hist.append(((i + 1) * dt, (g((i + 1) * dt) * uSV + Ex * yv).Evaluate()))
    return yv, yprev, hist

t0 = time.time()
dt = 2e-3
yT, yprev, hist = ars443(dt, T=0.75, frames=24)
print(f"{round(0.75 / dt)} ARS(4,4,3) steps: {time.time() - t0:.1f} s")

In [ ]:
umag = GridFunction(L2(mesh, order=1), multidim=0)
uframe = GridFunction(V)
for t, vec in hist:
    uframe.vec.data = vec
    tmp = GridFunction(L2(mesh, order=1)); tmp.Set(Norm(uframe)); umag.AddMultiDimComponent(tmp.vec)
Draw(umag, mesh, interpolate_multidim=True, animate=True, min=0, max=8, autoscale=False);

The jet leaves the gap at up to $|u| \approx 7$ while the inflow is at most 1.5: the local CFL rate $|u|/h$ is about 25 times larger in the gap than near the inlet. That is the case for multirate stepping. The constraint and the energy balance of the last step (trapezoidal rule for the power, so the defect also contains an $O(\\Delta t^2)$ quadrature part):

In [ ]:
uT = GridFunction(V); uT.vec.data = g(0.75) * uSV + Ex * yT
print(f"|div u(T)| = {sqrt(Integrate(div(uT)**2, mesh)):.1e}   (curl psi: divergence-free by construction)")

# semi-discrete energy identity for w = curl psi:  dE/dt = -a(w, w) - (C(u), w) - g' (u_S, w)
def power(yv, t):
    z = yv.CreateVector(); z.data = yv; z[Hp.ndof:] = 0         # facet unknowns from psi (facet-local solve)
    hfree = BitArray(X.ndof); hfree.Clear()
    for i in range(Hp.ndof, X.ndof): hfree[i] = X.FreeDofs()[i]
    rr = (A * z).Evaluate(); z.data -= A.Inverse(hfree, "sparsecholesky") * rr
    cz = z.CreateVector(); C(z, t, cz)
    return -InnerProduct(A * z, z) - InnerProduct(cz, z) - dg(t) * InnerProduct(fS, z), InnerProduct(A * z, z)
E0, E1 = 0.5 * InnerProduct(M * yprev, yprev), 0.5 * InnerProduct(M * yT, yT)
P0, D0 = power(yprev, 0.75 - dt); P1, D1 = power(yT, 0.75)
defect = (E1 - E0) - dt * (P0 + P1) / 2
print(f"last step: dE = {E1 - E0:.3e}, dt * power = {dt * (P0 + P1) / 2:.3e}, integrator defect / dissipation = {defect / (dt * (D0 + D1) / 2):.1e}")

## Measured comparison (fine mesh, 4632 triangles, $k = 3$)

The figures come from the experiment `stokes_channel.py` (raw data in `.json`, figures by a separate plotting script). Error: relative $L^2$ error of $u$ at $T = 0.75$ (decelerating flow) against a fine-$\Delta t$ reference on the same mesh. Wall times with 4 threads on an M1 Max; one stage solve of the $(\psi,\hat u)$ system costs 6.5 ms, one convection apply 21 ms, one solve of the mixed form with UMFPACK 194 ms.

![|u| of the reference at t = 0.25, 0.5, 0.75 and the CFL rate of each element](figures/flow_ns.png)

### Stokes (Re 1)

![Stokes: error at T against dt and wall time](figures/stokes_err.png)

All four methods are unconditionally stable and need one factorization (two for BDF2: its BDF1 start). The pulse is slow compared with the viscous relaxation, so a few steps suffice: Crank–Nicolson reaches $7\cdot10^{-6}$ with 12 steps (0.4 s, about half of it the factorization), BDF2 $4\cdot10^{-6}$ with 24 steps (0.7 s). SDIRK3 gains nothing over the second-order methods: stage order 1, observed order 2.3–2.8 on this stiff problem with time-dependent forcing. An explicit viscous step would have to stay below $1.5\cdot10^{-6}$ in every part of the channel (RK4, figure below), $4\cdot10^4$ times the step that gives $10^{-5}$. With the mixed form and UMFPACK every solve costs 30 times more (column "mixed" of the experiment's tables), the ranking is the same.

### Navier–Stokes (Re 250): stable steps and the cost of implicit convection

![largest stable steps](figures/stable_dt.png)

* Viscosity has to be implicit everywhere: explicit RK4 would need $\Delta t < 4.2\cdot10^{-4}$ in every region (the transition elements next to the refined box set the limit of the "coarse" region).
* IMEX with explicit upwind convection: ARS(4,4,3) is stable up to $\Delta t = 1.18\cdot10^{-3}$, ARS(2,2,2) up to $7.3\cdot10^{-4}$, SBDF2 only up to $4.4\cdot10^{-4}$ (its extrapolated convection is weakly unstable for upwind DG of order 3 near the imaginary axis).
* Fully implicit BDF2 with extrapolated convecting velocity (an Oseen system per step, GMRES preconditioned by the factored BDF2-Stokes matrix) is stable at any step but converges slowly: 88 GMRES iterations per step at $\Delta t = 0.0375$ (error 0.19), 28 at $\Delta t = 4.7\cdot10^{-3}$ (error $3\cdot10^{-3}$, 102 s).


### Navier–Stokes (Re 250): error against wall time

![Navier-Stokes: error at T against wall time](figures/ns_err.png)

| method | step | error at $T$ | wall time | global solves / convection applies |
|---|---|---|---|---|
| ARS(4,4,3), IMEX | $1.18\cdot10^{-3}$ (stability limit) | $1.8\cdot10^{-6}$ | 61 s | 2540 / 2540 |
| SBDF2, IMEX | $4.38\cdot10^{-4}$ (stability limit) | $4.4\cdot10^{-5}$ | 57 s | 1713 / 1713 |
| BDF2-Oseen, implicit | $4.7\cdot10^{-3}$ | $3.1\cdot10^{-3}$ | 102 s | 4500 / 4820 |
| BDF2-Oseen, implicit | $1.2\cdot10^{-3}$ | $1.8\cdot10^{-4}$ | 176 s | 8539 / 9819 |
| multirate, global projection | $H = 2.3\cdot10^{-3}$ | $9.2\cdot10^{-4}$ | 88 s | 5120 / 2322 |
| multirate, projection on the fast patch | $H = 2.3\cdot10^{-3}$ | $1.3\cdot10^{-3}$ | 100 s | 1280 / 2844 |
| multirate, no projection in the substeps | $H = 2.3\cdot10^{-3}$ | $1.6\cdot10^{-1}$ | 84 s | 1280 / 2844 |

Wall times $\pm 30$ % (other jobs ran on the machine); the counts are exact (applies in full-mesh equivalents; for BDF2-Oseen the solves are GMRES preconditioner applications).

* **IMEX wins at every accuracy.** The explicit convection step is limited by stability ($|u|/h$ in the gap), not by accuracy: at its limit ARS(4,4,3) is already at $1.8\cdot10^{-6}$. SBDF2 (one solve and one convection apply per step) is cheaper if $4.4\cdot10^{-5}$ is enough; below that ARS(4,4,3).
* **Fully implicit** BDF2-Oseen is stable at any step, but the Oseen systems cost 13 GMRES iterations per step at $\Delta t = 1.2\cdot10^{-3}$ (up to 88 at $\Delta t = 0.0375$) with the factored Stokes matrix as preconditioner, and BDF2 is less accurate: a larger step only pays at errors of $10^{-2}$.
* **Multirate** (Strang splitting: convection on the 40 % fastest elements by 4 SSP-RK3 substeps per coarse step, everything else by an ARS(4,4,3) step) is stable up to $H = 2.75\cdot10^{-3}$, 2.3 times the single-rate step, but 100–1000 times less accurate and more expensive per unit time, since every coarse step still contains the four global stage solves. On a coarse mesh the same scheme with *every* element "fast" has the same error: the loss comes from integrating convection apart from the implicit Stokes step, not from the substeps. Projecting the substeps only on the fast patch, with frozen fluxes through its boundary, is as accurate as the global projection; without any projection in the substeps the scheme drops to first order ($\|{\rm div}\,u\|$ up to 1.6 before the next coarse stage).

![cost per unit simulated time, split into solves, factorizations and explicit applies](figures/cost.png)

The integrator energy defect $E(t) - E(0) - \int_0^t P$ (composite Simpson on the step values) relative to the dissipated energy: BDF2 and BDF2-Oseen remove energy ($O(\Delta t^2)$), ARS(4,4,3) is neutral to $2\cdot10^{-6}$, the projected multirate variants add 1–3 %, the unprojected one removes 6–11 %.

![integrator energy defect](figures/energy.png)

In [ ]:
# tests
assert diff_mixed < 1e-9
assert sqrt(Integrate(div(uT)**2, mesh)) < 1e-9
assert np.isfinite(E1) and E1 < 10
assert abs(defect) < 0.05 * dt * (D0 + D1) / 2 and abs(E1 - 1.3115) < 0.01   # E(T) of a fine SBDF2 run (dt = 5e-4)
print("all tests passed")

## Summary

* **Stokes (Re 1): fully implicit with one factorization**, $\Delta t$ chosen by accuracy (Crank–Nicolson or BDF2): 12–24 steps for $10^{-5}$ over the pulse, under one second. There is no local time scale to resolve: the pressure is global and the viscous modes in the constriction are stiff, not fast.
* **Navier–Stokes (Re 250): IMEX at the convective stability limit**, viscosity implicit everywhere (explicit RK4 would need $\Delta t < 4.2\cdot10^{-4}$ in every region): SBDF2 for errors around $4.4\cdot10^{-5}$ (57 s), ARS(4,4,3) for $1.8\cdot10^{-6}$ (61 s). Neither the fully implicit Oseen step nor multirate convection pays.
* The stream-function form of the H(div)-HDG method gives the same velocities as BDM$_k$ $\times$ P$_{k-1}$ with half the unknowns and SPD solves (6.5 ms per solve). In 3D the mixed form (or a vector potential) is needed; with the mixed-form solve costs (UMFPACK, 194 ms per solve) a cost model of the same runs keeps the ranking: SBDF2 380 s, ARS(4,4,3) 540 s, BDF2-Oseen at least 940 s, multirate at least 670 s.

**References:** C. Lehrenfeld, J. Schöberl, CMAME 307 (2016) (H(div)-HDG); U. M. Ascher, S. J. Ruuth, R. J. Spiteri, APNUM 25 (1997) (IMEX Runge–Kutta); A. Sandu, SINUM 57 (2019) (multirate infinitesimal GARK); R. L. Speth, W. H. Green, S. MacNamara, G. Strang, SISC 35 (2013) (balanced splitting).